# 13 · Índice de presión sobre biodiversidad

**Objetivo:** Integrar pérdida de vegetación, borde forestal y superficie construida.

**Datos:** Sentinel-2, Dynamic World y SRTM.

**Relevancia para política ambiental y social:** Sirve para priorizar vigilancia, restauración y corredores.

**Limitaciones:** Los pesos son demostrativos y requieren validación participativa y ecológica.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
s2 = s2_composite("2025-01-01","2025-12-31")
ndvi_pressure = ee.Image(1).subtract(s2.normalizedDifference(["B8","B4"]).unitScale(0.1,0.8).clamp(0,1))
dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi).filterDate("2025-01-01","2025-12-31").select("label").mode()
built = dw.eq(6)
forest = dw.eq(1)
edge = forest.And(forest.focalMin(100,"circle","meters").Not())
pressure = ndvi_pressure.multiply(0.5).add(built.multiply(0.3)).add(edge.multiply(0.2)).rename("pressure").clip(aoi)
Map.addLayer(pressure, {"min":0,"max":1,"palette":["green","yellow","red"]}, "Presión")
Map
